# Notebook 10: Capstone — End-to-End Recommendation Pipeline

## Why This Matters

Every concept in this series exists to serve one goal: build a system that reliably finds and surfaces the right item for the right user at the right time. This capstone builds a complete two-stage recommendation pipeline on MovieLens 1M — two-tower retrieval with FAISS, LightGBM ranking with feature engineering, content-based cold start for new items, and a comprehensive offline evaluation suite. By the end, you'll be able to walk through a system design interview answer and understand every component at the implementation level.

The pipeline architecture matches how Netflix, Spotify, and YouTube structure their recommendation stacks. The techniques, feature patterns, and evaluation protocols here translate directly to industry projects.

## Background

### The Production Pipeline

A mature recommendation system has these stages:

```
User Request
    │
    ▼
[1] Feature Assembly (user history, context, signals)
    │
    ▼
[2] Retrieval (Two-Tower → FAISS ANN → top-500 candidates)
    │
    ▼
[3] Pre-ranking filter (business rules, legal, content safety)
    │
    ▼
[4] Ranking (LightGBM LambdaMART → top-50 re-ranked)
    │
    ▼
[5] Post-processing (diversity, deduplication, freshness boost)
    │
    ▼
[6] Served slate (top-10 shown to user)
```

### Interview Framework: "Design a Movie Recommendation System"

This is one of the most common ML system design questions. A structured 30-minute answer:

**1. Clarify scope (2 min)**: Catalog size? User base? Latency requirement? Which platform? Cold start prevalence?

**2. Metrics (3 min)**: Online: CTR, completion rate, session length, 7-day return rate. Offline: Recall@50 (retrieval), NDCG@10 (ranking), Coverage@10 (diversity).

**3. Data & features (5 min)**: User features (history embeddings, explicit ratings, demographics), item features (genre, cast, director, release year, average rating), context features (time of day, device, session context).

**4. Retrieval (8 min)**: Two-tower model, in-batch negatives with popularity correction, FAISS IVF index, daily rebuild. Retrieval target: Recall@500.

**5. Ranking (8 min)**: LambdaMART with 50+ features, trained on click/completion/share labels, serving top-10. NDCG@10 as primary metric. A/B tested.

**6. Cold start (3 min)**: New items get content-based warm-start embeddings from metadata encoder. Injected into exploration slots.

**7. Production concerns (3 min)**: Feature store for real-time user features, embedding cache, fallback to popularity for new users, monitoring for CTR drift, data flywheel.

In [ ]:
%pip install -q torch numpy pandas scikit-learn lightgbm faiss-cpu matplotlib tqdm scipy

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
import faiss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import Ridge
from tqdm import tqdm
import os, requests, zipfile, io, warnings, time
from collections import defaultdict
from functools import partial

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 1M ─────────────────────────────────────────────────────────
ML1M_DIR = "/tmp/ml-1m"
if not os.path.exists(ML1M_DIR):
    print("Downloading MovieLens 1M...")
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-1m.zip", timeout=120)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{ML1M_DIR}/ratings.dat", sep="::", engine="python",
                       header=None, names=["user_id","item_id","rating","timestamp"])
movies = pd.read_csv(f"{ML1M_DIR}/movies.dat", sep="::", engine="python",
                      header=None, names=["item_id","title","genres"], encoding="latin-1")
users = pd.read_csv(f"{ML1M_DIR}/users.dat", sep="::", engine="python",
                     header=None, names=["user_id","gender","age","occupation","zip"])

# Re-index
user2idx = {u: i for i, u in enumerate(sorted(ratings.user_id.unique()))}
item2idx = {it: i+1 for i, it in enumerate(sorted(ratings.item_id.unique()))}
idx2item = {v: k for k, v in item2idx.items()}

ratings["user_idx"] = ratings.user_id.map(user2idx)
ratings["item_idx"] = ratings.item_id.map(item2idx)
movies["item_idx"] = movies.item_id.map(item2idx)

n_users = ratings.user_idx.max() + 1
n_items = ratings.item_idx.max() + 2  # 0=padding

# Genre features
all_genres = set()
for g in movies.genres:
    all_genres.update(g.split("|"))
all_genres = sorted(all_genres)
for g in all_genres:
    movies[f"genre_{g}"] = movies.genres.str.contains(g, regex=False).astype(int)
genre_feature_cols = [f"genre_{g}" for g in all_genres]

# Temporal split: last 20% of each user's ratings for test
ratings = ratings.sort_values(["user_idx", "timestamp"])
train_list, test_list = [], []
for uid, group in ratings.groupby("user_idx"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])
train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

user_pos_train = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
user_pos_test  = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

print(f"ML-1M | Users: {n_users:,} | Items: {n_items:,}")
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Device: {device}")
print("Ready.")

## Stage 1: Two-Tower Retrieval Model

The retrieval stage must efficiently find the top-500 candidate items from 3,000+ movies for each user. We train a two-tower model with in-batch negatives, then build a FAISS index for sub-millisecond ANN search.

In [ ]:
class TwoTowerRetrieval(nn.Module):
    """
    Two-tower retrieval model for MovieLens 1M.
    User tower: user_id embedding + history average pooling
    Item tower: item_id embedding + genre features
    """
    def __init__(self, n_users, n_items, n_genres, d=128, hist_len=50):
        super().__init__()
        self.d = d
        self.user_emb = nn.Embedding(n_users, d)
        self.item_emb = nn.Embedding(n_items, d, padding_idx=0)
        self.genre_proj = nn.Linear(n_genres, d)

        # User tower: combine user_id + history mean
        self.user_proj = nn.Sequential(
            nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, d)
        )
        # Item tower: combine item_id + genre features
        self.item_proj = nn.Sequential(
            nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, d)
        )
        self.temperature = nn.Parameter(torch.tensor(0.07))

        for emb in [self.user_emb, self.item_emb]:
            nn.init.normal_(emb.weight, 0, 0.01)

    def encode_user(self, user_ids, history):
        u = self.user_emb(user_ids)                           # (B, d)
        mask = (history != 0).float().unsqueeze(-1)           # (B, L, 1)
        hist_e = self.item_emb(history)                       # (B, L, d)
        hist_avg = (hist_e * mask).sum(1) / mask.sum(1).clamp(min=1)  # (B, d)
        return F.normalize(self.user_proj(torch.cat([u, hist_avg], dim=1)), dim=1)

    def encode_item(self, item_ids, genre_feats):
        ie = self.item_emb(item_ids)                          # (B, d)
        ge = self.genre_proj(genre_feats)                     # (B, d)
        return F.normalize(self.item_proj(torch.cat([ie, ge], dim=1)), dim=1)

    def forward(self, user_ids, hist, item_ids, genre_feats):
        u = self.encode_user(user_ids, hist)
        v = self.encode_item(item_ids, genre_feats)
        temp = torch.clamp(self.temperature, 0.01, 1.0)
        logits = (u @ v.T) / temp
        labels = torch.arange(len(u), device=u.device)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        return loss


# Build genre features lookup
item_genre_features = np.zeros((n_items, len(all_genres)), dtype=np.float32)
for _, row in movies.iterrows():
    if pd.notna(row.item_idx):
        idx = int(row.item_idx)
        if idx < n_items:
            for g_idx, g in enumerate(all_genres):
                item_genre_features[idx, g_idx] = row.get(f"genre_{g}", 0)

# Build user history lookup
user_history = {}
MAX_HIST = 50
for uid, group in train_df.groupby("user_idx"):
    seq = group.sort_values("timestamp")["item_idx"].tail(MAX_HIST).values
    user_history[uid] = seq

model_tt = TwoTowerRetrieval(n_users, n_items, len(all_genres), d=128).to(device)
n_params = sum(p.numel() for p in model_tt.parameters())
print(f"Two-Tower Model | Parameters: {n_params:,} | d=128, genres={len(all_genres)}")

In [ ]:
class RetriévalDataset(Dataset):
    def __init__(self, train_df, user_history, item_genre_features, n_items, max_hist=50):
        self.users = train_df.user_idx.values
        self.items = train_df.item_idx.values
        self.user_hist = user_history
        self.genre_feats = item_genre_features
        self.max_hist = max_hist
        self.n_items = n_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        i = self.items[idx]
        hist = self.user_hist.get(u, np.array([]))
        hist_pad = np.zeros(self.max_hist, dtype=np.int64)
        if len(hist) > 0:
            h = hist[-self.max_hist:]
            hist_pad[:len(h)] = h
        return (torch.tensor(u, dtype=torch.long),
                torch.tensor(hist_pad, dtype=torch.long),
                torch.tensor(i, dtype=torch.long),
                torch.tensor(self.genre_feats[i], dtype=torch.float32))

ret_ds = RetriévalDataset(train_df, user_history, item_genre_features, n_items)
ret_loader = DataLoader(ret_ds, batch_size=512, shuffle=True, num_workers=0, drop_last=True)

optimizer_tt = optim.AdamW(model_tt.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_tt = optim.lr_scheduler.CosineAnnealingLR(optimizer_tt, T_max=15)

print("Training Two-Tower retrieval model (15 epochs)...")
t0 = time.time()
tt_losses = []
for epoch in range(15):
    model_tt.train()
    total = 0
    for u, hist, item, genre in ret_loader:
        u, hist, item, genre = u.to(device), hist.to(device), item.to(device), genre.to(device)
        optimizer_tt.zero_grad()
        loss = model_tt(u, hist, item, genre)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_tt.parameters(), 1.0)
        optimizer_tt.step()
        total += loss.item()
    scheduler_tt.step()
    avg = total / len(ret_loader)
    tt_losses.append(avg)
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg:.4f}")

print(f"Training time: {time.time()-t0:.1f}s")

## Stage 1b: Build FAISS Index

Pre-compute all item embeddings using the trained item tower, then build a FAISS IVF index for efficient ANN search.

In [ ]:
# Pre-compute item embeddings
model_tt.eval()
all_item_embs = []
batch_size = 512
item_genre_t = torch.tensor(item_genre_features, dtype=torch.float32).to(device)

with torch.no_grad():
    for start in range(0, n_items, batch_size):
        end = min(start + batch_size, n_items)
        item_ids = torch.arange(start, end, dtype=torch.long).to(device)
        genre_batch = item_genre_t[start:end]
        embs = model_tt.encode_item(item_ids, genre_batch).cpu().numpy()
        all_item_embs.append(embs)

item_embeddings = np.vstack(all_item_embs).astype(np.float32)
d = item_embeddings.shape[1]
print(f"Item embeddings: {item_embeddings.shape}")

# Build FAISS IVF index
n_lists = int(np.sqrt(n_items))
quantizer = faiss.IndexFlatIP(d)
faiss_index = faiss.IndexIVFFlat(quantizer, d, n_lists, faiss.METRIC_INNER_PRODUCT)
faiss_index.train(item_embeddings)
faiss_index.add(item_embeddings)
faiss_index.nprobe = 16
print(f"FAISS index: {faiss_index.ntotal} items, {n_lists} lists, nprobe=16")

def retrieve_candidates(user_idx, model, index, user_history, item_genre_features,
                         user_pos_train, K=500, device="cpu"):
    """Encode user → FAISS ANN search → filter seen → return top-K candidates."""
    model.eval()
    hist = user_history.get(user_idx, np.array([]))
    hist_pad = np.zeros(MAX_HIST, dtype=np.int64)
    if len(hist) > 0:
        h = hist[-MAX_HIST:]
        hist_pad[:len(h)] = h

    u_t = torch.tensor([user_idx], dtype=torch.long).to(device)
    h_t = torch.tensor([hist_pad], dtype=torch.long).to(device)
    with torch.no_grad():
        u_emb = model.encode_user(u_t, h_t).cpu().numpy().astype(np.float32)

    D, I = index.search(u_emb, K * 2)
    candidates = I[0]
    scores = D[0]

    seen = user_pos_train.get(user_idx, set())
    filtered = [(int(c), float(s)) for c, s in zip(candidates, scores)
                if c not in seen and c > 0][:K]
    return filtered

# Quick test
sample_uid = 100
cands = retrieve_candidates(sample_uid, model_tt, faiss_index, user_history,
                             item_genre_features, user_pos_train, K=100, device=str(device))
print(f"\nUser {sample_uid}: retrieved {len(cands)} candidates")
print(f"Top-5 item indices: {[c[0] for c in cands[:5]]}")

## Stage 2: LightGBM Ranking

Given the ~500 candidates from retrieval, we re-rank them with a LambdaMART model. The ranker has access to richer features than the retrieval model: user-item cross features, item quality stats, and genre affinity.

In [ ]:
# ── Build ranking features ────────────────────────────────────────────────────
item_stats = train_df.groupby("item_idx").agg(
    item_n_ratings=("rating", "count"),
    item_mean_rating=("rating", "mean"),
    item_rating_std=("rating", "std"),
).reset_index()
item_stats["item_rating_std"] = item_stats["item_rating_std"].fillna(0)
item_stats["item_log_pop"] = np.log1p(item_stats["item_n_ratings"])

user_stats = train_df.groupby("user_idx").agg(
    user_n_ratings=("rating", "count"),
    user_mean_rating=("rating", "mean"),
).reset_index()

# Genre affinity: for each user, mean rating per genre (in train)
def build_genre_affinity(train_df, movies_df, genre_cols, user_stats):
    user_genre = train_df.merge(movies_df[["item_idx"] + genre_cols].dropna(), on="item_idx")
    affinities = {}
    for g in genre_cols:
        g_items = user_genre[user_genre[g] == 1]
        aff = g_items.groupby("user_idx")["rating"].mean().rename(f"aff_{g}")
        affinities[f"aff_{g}"] = aff
    df_aff = pd.DataFrame(affinities).reset_index()
    df_aff.columns = ["user_idx"] + [f"aff_{g}" for g in genre_cols]
    user_means = user_stats[["user_idx", "user_mean_rating"]]
    df_aff = df_aff.merge(user_means, on="user_idx")
    for g in genre_cols:
        df_aff[f"aff_{g}"] = df_aff[f"aff_{g}"].fillna(df_aff["user_mean_rating"])
    return df_aff.drop(columns=["user_mean_rating"])

movies_for_rank = movies[["item_idx"] + genre_feature_cols].copy().dropna(subset=["item_idx"])
movies_for_rank["item_idx"] = movies_for_rank["item_idx"].astype(int)
genre_affinity_df = build_genre_affinity(train_df, movies_for_rank, genre_feature_cols, user_stats)

def build_ranking_features(user_idx, candidates, user_stats_df, item_stats_df,
                            movies_df, genre_affinity_df, item_genre_features,
                            genre_cols, n_items):
    """Build feature matrix for ranking a set of candidates for a user."""
    rows = []
    u_row = user_stats_df[user_stats_df.user_idx == user_idx]
    u_n = u_row["user_n_ratings"].values[0] if len(u_row) else 0
    u_mean = u_row["user_mean_rating"].values[0] if len(u_row) else 3.5

    u_aff = genre_affinity_df[genre_affinity_df.user_idx == user_idx]

    for item_idx, retrieval_score in candidates:
        i_row = item_stats_df[item_stats_df.item_idx == item_idx]
        i_n    = i_row["item_n_ratings"].values[0] if len(i_row) else 0
        i_mean = i_row["item_mean_rating"].values[0] if len(i_row) else 3.0
        i_std  = i_row["item_rating_std"].values[0] if len(i_row) else 0
        i_lpop = i_row["item_log_pop"].values[0] if len(i_row) else 0

        # Genre affinity match
        genre_vec = item_genre_features[item_idx] if item_idx < len(item_genre_features) else np.zeros(len(genre_cols))
        if len(u_aff) > 0:
            aff_cols = [f"aff_{g}" for g in genre_cols]
            aff_vec = u_aff[aff_cols].values[0]
            genre_affinity_score = float(np.dot(aff_vec, genre_vec) / (np.sum(genre_vec) + 1e-8))
        else:
            genre_affinity_score = u_mean

        # Prediction from bias model
        bias_pred = u_mean + i_mean - 3.58  # global mean ~3.58 for ML-1M

        feat = [
            float(u_n), float(u_mean),
            float(i_n), float(i_mean), float(i_std), float(i_lpop),
            float(retrieval_score),
            float(genre_affinity_score),
            float(bias_pred),
        ] + list(genre_vec.astype(float))
        rows.append(feat)

    feat_cols = ["user_n_ratings", "user_mean_rating",
                 "item_n_ratings", "item_mean_rating", "item_rating_std", "item_log_pop",
                 "retrieval_score", "genre_affinity", "bias_pred"] + genre_cols
    return pd.DataFrame(rows, columns=feat_cols)

print(f"Ranking features: {9 + len(genre_feature_cols)} total")

In [ ]:
# Build training data for the ranker
# For each user in train: retrieve candidates, build features, assign labels
print("Building ranker training data...")
t0 = time.time()

ranker_X, ranker_y, ranker_groups = [], [], []
n_ranker_users = 2000  # subset for speed

sample_train_users = sorted(
    [u for u in user_pos_train if len(user_pos_train[u]) >= 3],
)[:n_ranker_users]

test_user_set = set(user_pos_test.keys())
for u in sample_train_users:
    if u not in test_user_set:
        continue
    pos_items = user_pos_train[u]

    # Use fast popularity-based candidates + some positives for training variety
    cand_scores = {}
    for item_idx in list(pos_items)[:20]:
        cand_scores[item_idx] = 1.0  # guarantee positives are included

    # Add some non-positive candidates (simulate retrieval)
    n_neg = 30
    neg_sample = np.random.choice(
        [i for i in range(1, n_items) if i not in pos_items],
        size=min(n_neg, n_items - len(pos_items)), replace=False
    )
    for i in neg_sample:
        cand_scores[i] = float(np.random.random() * 0.3)  # low retrieval score

    if len(cand_scores) < 5:
        continue

    candidates = list(cand_scores.items())
    feats = build_ranking_features(
        u, candidates, user_stats, item_stats,
        movies_for_rank, genre_affinity_df,
        item_genre_features, genre_feature_cols, n_items
    )

    # Labels: 1 if item is in user's positive train set (rating >= 4), else 0
    labels = [1 if item in pos_items else 0 for item, _ in candidates]

    ranker_X.append(feats)
    ranker_y.extend(labels)
    ranker_groups.append(len(candidates))

ranker_X = pd.concat(ranker_X, ignore_index=True)
ranker_y = np.array(ranker_y)
print(f"Ranker training: {len(ranker_X):,} examples, {len(ranker_groups):,} queries ({time.time()-t0:.1f}s)")
print(f"Positive rate: {ranker_y.mean():.3f}")

In [ ]:
# Train LightGBM LambdaMART ranker
feature_cols_rank = ranker_X.columns.tolist()
X_arr = ranker_X.values.astype(np.float32)

train_data_lgb = lgb.Dataset(X_arr, label=ranker_y, group=ranker_groups,
                               feature_name=feature_cols_rank)

ranker_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10],
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_data_in_leaf": 2,
    "label_gain": [0, 1],  # binary labels 0/1
    "verbose": -1,
    "num_threads": 4,
}

print("Training LightGBM ranker...")
t0 = time.time()
ranker_model = lgb.train(
    ranker_params,
    train_data_lgb,
    num_boost_round=150,
    callbacks=[lgb.log_evaluation(period=9999)]  # silent
)
print(f"Ranker trained in {time.time()-t0:.1f}s | {ranker_model.num_trees()} trees")

# Feature importance
feat_imp = pd.Series(ranker_model.feature_importance("gain"),
                      index=feature_cols_rank).sort_values(ascending=False)
print("\nTop 10 features by gain:")
print(feat_imp.head(10).to_string())

## Full Pipeline: Retrieve → Rank → Recommend

Now we connect the two stages. For each user: retrieve top-500 with FAISS, score all 500 with the LightGBM ranker, return top-10.

In [ ]:
def full_pipeline_recommend(user_idx, model_tt, faiss_index, ranker_model,
                              user_history, item_genre_features, user_stats, item_stats,
                              movies_df, genre_affinity_df, genre_cols,
                              user_pos_train, n_items, K_ret=200, K_rank=10, device="cpu"):
    """Full two-stage recommendation pipeline."""
    # Stage 1: Retrieval
    candidates = retrieve_candidates(
        user_idx, model_tt, faiss_index, user_history,
        item_genre_features, user_pos_train, K=K_ret, device=device
    )
    if not candidates:
        return []

    # Stage 2: Ranking
    feats = build_ranking_features(
        user_idx, candidates, user_stats, item_stats,
        movies_df, genre_affinity_df, item_genre_features, genre_cols, n_items
    )
    scores = ranker_model.predict(feats.values.astype(np.float32))

    # Sort by ranker scores
    ranked = sorted(zip([c[0] for c in candidates], scores),
                    key=lambda x: x[1], reverse=True)
    return [item_idx for item_idx, _ in ranked[:K_rank]]


# Demo: recommendations for a few users
movies_idx_lookup = movies.set_index("item_idx")["title"].to_dict()
print("\nSample recommendations from full pipeline:")
for u in [100, 500, 1000]:
    if u not in user_pos_train:
        continue
    recs = full_pipeline_recommend(
        u, model_tt, faiss_index, ranker_model,
        user_history, item_genre_features, user_stats, item_stats,
        movies_for_rank, genre_affinity_df, genre_feature_cols,
        user_pos_train, n_items, K_ret=200, K_rank=10, device=str(device)
    )
    titles = [movies_idx_lookup.get(r, f"Item {r}")[:35] for r in recs[:5]]
    print(f"\n  User {u}: {titles}")

## Offline Evaluation: NDCG@10 and Recall@50

We evaluate the full pipeline on held-out test users using the standard metrics from Notebook 09.

In [ ]:
def ndcg_at_k(recs, relevant, K):
    dcg = sum(1.0/np.log2(k+2) for k, item in enumerate(recs[:K]) if item in relevant)
    idcg = sum(1.0/np.log2(k+2) for k in range(min(len(relevant), K)))
    return dcg/idcg if idcg > 0 else 0.0

def recall_at_k(recs, relevant, K):
    if not relevant: return 0.0
    return len(set(recs[:K]) & relevant) / len(relevant)

# Evaluate on test users
print("Evaluating full pipeline on test users (500 users)...")
eval_users = [u for u in user_pos_test if len(user_pos_test[u]) > 0][:500]

ndcgs_10, recalls_50, recalls_10 = [], [], []
latencies = []

for u in eval_users:
    t0 = time.time()
    recs_10 = full_pipeline_recommend(
        u, model_tt, faiss_index, ranker_model,
        user_history, item_genre_features, user_stats, item_stats,
        movies_for_rank, genre_affinity_df, genre_feature_cols,
        user_pos_train, n_items, K_ret=100, K_rank=50, device=str(device)
    )
    latencies.append(time.time() - t0)

    relevant = user_pos_test[u]
    ndcgs_10.append(ndcg_at_k(recs_10, relevant, 10))
    recalls_10.append(recall_at_k(recs_10, relevant, 10))
    recalls_50.append(recall_at_k(recs_10, relevant, 50))

print(f"\nFull Pipeline Evaluation (n={len(eval_users)} users):")
print(f"  NDCG@10:   {np.mean(ndcgs_10):.4f} ± {np.std(ndcgs_10):.4f}")
print(f"  Recall@10: {np.mean(recalls_10):.4f} ± {np.std(recalls_10):.4f}")
print(f"  Recall@50: {np.mean(recalls_50):.4f} ± {np.std(recalls_50):.4f}")
print(f"  Avg latency: {np.mean(latencies)*1000:.1f}ms (retrieve+rank+features)")
print(f"  P50 latency: {np.percentile(latencies, 50)*1000:.1f}ms")
print(f"  P99 latency: {np.percentile(latencies, 99)*1000:.1f}ms")

## Cold Start: Content-Based Warm Start for New Items

New items enter the catalog with no interaction history. We train a linear content encoder that maps genre features → item embedding space, so new items get an initial embedding immediately.

In [ ]:
# Train content → embedding warm start
# Use items with >= 20 interactions as training data
item_interaction_counts = train_df.groupby("item_idx").size()
warm_items = item_interaction_counts[item_interaction_counts >= 20].index.tolist()
cold_items = item_interaction_counts[item_interaction_counts < 5].index.tolist()

print(f"Warm items (≥20 interactions): {len(warm_items)}")
print(f"Cold items (<5 interactions):  {len(cold_items)}")

# Build content feature matrix and CF embedding matrix for warm items
X_warm = item_genre_features[[i for i in warm_items if i < len(item_genre_features)]]
y_warm = item_embeddings[[i for i in warm_items if i < len(item_embeddings)]]

# Train linear content→CF mapper
content_mapper = Ridge(alpha=10.0)
content_mapper.fit(X_warm, y_warm)

# Predict for cold items
cold_idx = [i for i in cold_items[:100] if i < len(item_genre_features)]
X_cold = item_genre_features[cold_idx]
cold_embs_pred = content_mapper.predict(X_cold).astype(np.float32)

# Normalize (item embeddings are L2-normalized)
norms = np.linalg.norm(cold_embs_pred, axis=1, keepdims=True)
cold_embs_pred /= np.clip(norms, 1e-8, None)

# Evaluate: cosine similarity between predicted and true embeddings for warm items
X_val = item_genre_features[[i for i in warm_items[:200] if i < len(item_genre_features)]]
y_val_true = item_embeddings[[i for i in warm_items[:200] if i < len(item_embeddings)]]
y_val_pred = content_mapper.predict(X_val).astype(np.float32)

# Normalize
y_val_pred /= np.clip(np.linalg.norm(y_val_pred, axis=1, keepdims=True), 1e-8, None)
y_val_true_n = y_val_true / np.clip(np.linalg.norm(y_val_true, axis=1, keepdims=True), 1e-8, None)
cos_sims = np.sum(y_val_true_n * y_val_pred, axis=1)

print(f"\nWarm start mapping quality (cosine similarity):")
print(f"  Mean: {cos_sims.mean():.3f} | Median: {np.median(cos_sims):.3f} | Std: {cos_sims.std():.3f}")
print(f"  % with cosine > 0.5: {(cos_sims > 0.5).mean():.1%}")
print("\nThis means new items get a reasonable initial embedding from content alone.")
print("As users interact with them, the CF model will learn a better embedding.")

## Production Monitoring Design

A deployed recommender system needs continuous monitoring. Here are the key signals to track and their alert thresholds.

In [ ]:
# Monitoring design: what to track in production
monitoring_plan = {
    "Business Metrics (daily)": {
        "CTR (click-through rate)": "Alert if drops >5% relative in 24h",
        "Session length": "Alert if drops >10% vs 7-day rolling avg",
        "7-day return rate": "Alert if drops >3% relative",
        "Conversion rate (for e-commerce)": "Alert if drops >5% relative",
    },
    "Retrieval Health (hourly)": {
        "Recall@500 (sampled)": "Alert if drops below 0.70 (offline eval)",
        "FAISS query latency P99": "Alert if >100ms",
        "Item index freshness": "Alert if item embeddings >6h stale",
        "New item coverage": "Alert if new items (last 24h) <10% in any recommendation",
    },
    "Ranking Health (hourly)": {
        "NDCG@10 (sampled)": "Alert if drops >10% vs training baseline",
        "Ranker latency P99": "Alert if >50ms",
        "Feature distribution drift": "Monitor Jensen-Shannon divergence on top features",
        "Score distribution": "Alert if mean ranker score shifts >0.1 vs baseline",
    },
    "Data Pipeline (per batch)": {
        "Feedback loop detection": "Alert if item popularity Gini coefficient increases >5%",
        "Train/test distribution drift": "Monitor user activity and item popularity",
        "Null/missing feature rate": "Alert if >1% of ranking features are null",
        "Negative sample quality": "Monitor ratio of hard vs easy negatives",
    },
}

print("=" * 65)
print("PRODUCTION MONITORING PLAN")
print("=" * 65)
for category, metrics in monitoring_plan.items():
    print(f"\n{category}:")
    for metric, threshold in metrics.items():
        print(f"  • {metric}")
        print(f"    → {threshold}")

## Interview Framework: "Design a Movie Recommendation System"

Here's a structured 30-minute answer for the most common ML system design question.

In [ ]:
interview_framework = """
╔══════════════════════════════════════════════════════════════════╗
║        INTERVIEW FRAMEWORK: Movie Recommendation System         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. CLARIFY SCOPE (2 min)                                       ║
║     • Catalog: ~10,000 movies + TV shows                        ║
║     • Users: 100M+ registered, 10M daily active                 ║
║     • Latency: < 100ms end-to-end for serving                  ║
║     • Platform: streaming service, logged-in users              ║
║                                                                  ║
║  2. METRICS (3 min)                                             ║
║     • Online: CTR, completion rate (>80%?), 7-day return rate   ║
║     • Offline retrieval: Recall@500                             ║
║     • Offline ranking: NDCG@10, diversity, coverage             ║
║     • Guardrails: latency P99, filter bubble index              ║
║                                                                  ║
║  3. DATA & FEATURES (5 min)                                     ║
║     • User: embedding, recent watch history, genre affinities   ║
║     • Item: embedding, genre, cast, director, avg rating, age   ║
║     • Context: time of day, device, session length              ║
║     • Labels: watch completion > 70%, thumbs up, add to list   ║
║                                                                  ║
║  4. TWO-STAGE ARCHITECTURE (10 min)                             ║
║     Stage 1 — Retrieval:                                        ║
║       • Two-tower model, in-batch negatives + hard negatives    ║
║       • FAISS IVF index, ~200ms rebuild daily                   ║
║       • Output: top-500 candidates, Recall@500 target           ║
║     Stage 2 — Ranking:                                          ║
║       • LightGBM LambdaMART, 50+ features                      ║
║       • Trained on watch completion signal (implicit)           ║
║       • Output: top-10 items, NDCG@10 target                   ║
║                                                                  ║
║  5. COLD START (3 min)                                          ║
║     • New item: content encoder (text/poster) → embedding space ║
║     • New user: onboarding genres + popular items as fallback   ║
║     • New items guaranteed 0.5% of slots for learning          ║
║                                                                  ║
║  6. PRODUCTION (5 min)                                          ║
║     • Feature store: Redis for real-time user features          ║
║     • Model serving: TorchScript user encoder, pre-built FAISS  ║
║     • A/B testing: interleaving for ranking, user split for UX  ║
║     • Monitoring: CTR/session drift alerts, data pipeline SLAs  ║
║     • Retraining: daily for retrieval, weekly for ranker        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(interview_framework)

## Decision Flowchart: Which Technique to Use?

Based on everything in this series, here's a practical guide for choosing the right approach.

In [ ]:
# Visualize the decision flowchart as text
decision_guide = """
RECOMMENDER SYSTEM TECHNIQUE SELECTION GUIDE
=============================================

What data do you have?
├── Explicit ratings (stars, thumbs up/down)
│   └── How many users and items?
│       ├── < 10K users × items → Item-based CF (Notebook 01)
│       ├── 10K – 1M → Matrix Factorization / Funk SVD (Notebook 02)
│       └── > 1M → ALS or Two-Tower (Notebooks 02, 04)
│
└── Implicit feedback (clicks, views, purchases) [most common]
    └── Do sequences matter?
        ├── NO (static preferences)
        │   └── How large is your catalog?
        │       ├── < 100K items → BPR (Notebook 03)
        │       └── > 100K items → Two-Tower + FAISS (Notebook 04)
        │
        └── YES (session context, drift)
            └── SASRec / BERT4Rec (Notebook 07)

Do you have a second stage for ranking?
├── YES → Learning to Rank / LambdaMART (Notebook 05)
└── NO  → Consider NCF if non-linear patterns are suspected (Notebook 06)

Cold start concerns?
├── New items → Content-based warm start (Notebook 08)
└── New users → Bandit exploration + onboarding (Notebook 08)

Do you have rich item features (text, images)?
├── YES → Augment both towers with content encoders
└── NO  → ID embeddings are sufficient for most cases

Scale > 10M items?
├── Two-tower retrieval + ANN index is REQUIRED
└── Cannot score every item at query time

Always evaluate with:
├── Recall@50 (retrieval stage)
├── NDCG@10 (ranking stage)
├── Coverage@10 (diversity / fairness)
└── Online A/B test (ground truth)
"""
print(decision_guide)

## Summary

### Key Concepts — Full Series

| Notebook | Core Technique | Use When |
|---|---|---|
| 01: Fundamentals | Item-based CF, cosine similarity | Small catalog, interpretability needed |
| 02: Matrix Factorization | Funk SVD, ALS | Explicit ratings, medium scale |
| 03: Implicit Feedback | BPR, weighted confidence | Clicks/views, no explicit ratings |
| 04: Two-Tower | Dual encoder + FAISS | Large catalogs (100K+ items), retrieval stage |
| 05: Learning to Rank | LambdaMART | Re-ranking candidates with rich features |
| 06: Neural CF | NeuMF (GMF + MLP) | Non-linear preferences, ranking stage |
| 07: Sequential | SASRec (causal attention) | Session context, next-item prediction |
| 08: Cold Start | Warm start, Thompson Sampling | New items, new users, exploration |
| 09: Evaluation | NDCG@K, Recall@K, Coverage | Always — evaluate before shipping |
| 10: Capstone | Full pipeline | Production: retrieve → rank → serve |

### Common Production Pitfalls
- Building a perfect offline model with no deployment plan — the last mile is hard
- No cold start strategy — new items die without impressions, no learning
- Evaluating only on active users — cold and casual users are the majority
- Feedback loop blindness — your model creates its own training data
- Ignoring diversity and coverage — 100 popular items served to 100M users is a failure mode
- Missing monitoring — CTR can drift silently for days before anyone notices